# 02 — ساخت ویژگی‌های انتقال‌پذیر از دیتاست CHARGED

## هدف نوت‌بوک

در این نوت‌بوک، داده‌های خام CHARGED به یک مجموعه‌دادهٔ تحلیلی در سطح ایستگاه-روز تبدیل می‌شوند. تمام ویژگی‌ها باید برای دیتاست اعتبارسنجی خارجی UrbanEV و شهر اصفهان نیز قابل استخراج یا برآورد باشند.

متغیر هدف از مجموع مدت شارژ ساعتی در فایل `duration.csv` ساخته می‌شود. ویژگی‌ها شامل ظرفیت ایستگاه، تراکم گروه‌های مفهومی POI در پیرامون ایستگاه، تقویم و شرایط جوی خواهند بود.

هیچ ویژگی‌ای که از تقاضای مشاهده‌شدهٔ آینده یا مقادیر تجمعی `total_duration`، `total_volume` و `avg_power` استخراج شده باشد، وارد مدل نخواهد شد.

In [1]:
# Prompt: Import libraries and define reproducible paths for CHARGED feature engineering.

from pathlib import Path
from io import BytesIO
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

candidate_roots = [Path.cwd().resolve(), Path.cwd().resolve().parent]

PROJECT_ROOT = next(
    (
        path
        for path in candidate_roots
        if (path / "data" / "raw" / "charged" / "Hourly.zip").exists()
    ),
    None,
)

assert PROJECT_ROOT is not None, (
    "Hourly.zip پیدا نشد. بررسی کن فایل در مسیر "
    "data/raw/charged/Hourly.zip قرار داشته باشد."
)

CHARGED_ARCHIVE_PATH = (
    PROJECT_ROOT / "data" / "raw" / "charged" / "Hourly.zip"
)

INTERIM_DATA_DIR = PROJECT_ROOT / "data" / "interim"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"
OUTPUT_FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"

for directory in [
    INTERIM_DATA_DIR,
    PROCESSED_DATA_DIR,
    OUTPUT_TABLES_DIR,
    OUTPUT_FIGURES_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

TRAINING_CITY_CODES = ["AMS", "JHB", "LOA", "MEL", "SPO"]
EXCLUDED_CITY_CODE = "SZH"

CITY_LOOKUP = {
    "AMS": {"city_name": "Amsterdam", "country_name": "Netherlands"},
    "JHB": {"city_name": "Johannesburg", "country_name": "South Africa"},
    "LOA": {"city_name": "Los Angeles", "country_name": "United States"},
    "MEL": {"city_name": "Melbourne", "country_name": "Australia"},
    "SPO": {"city_name": "Sao Paulo", "country_name": "Brazil"},
}

HOURLY_CHUNK_SIZE = 200

print(f"Project root: {PROJECT_ROOT}")
print(f"Training cities: {TRAINING_CITY_CODES}")
print(f"Excluded from training: {EXCLUDED_CITY_CODE}")
print(f"CHARGED archive size: {CHARGED_ARCHIVE_PATH.stat().st_size / (1024 ** 2):.2f} MB")

Project root: F:\UrbanEV_Charging_Demand
Training cities: ['AMS', 'JHB', 'LOA', 'MEL', 'SPO']
Excluded from training: SZH
CHARGED archive size: 256.27 MB


## ۲-۱. تعریف واحد تحلیل و متغیرهای هدف

واحد تحلیل در این پژوهش «ایستگاه-روز» است. داده‌های ساعتی `duration.csv` به سطح روز تجمیع می‌شوند تا تقاضای روزانهٔ هر ایستگاه محاسبه شود.

دو خروجی ساخته می‌شود:

- `daily_duration_hours`: مجموع مدت شارژ روزانه در هر ایستگاه
- `daily_duration_per_charger`: مدت شارژ روزانه، تقسیم بر تعداد پورت‌های شارژ ایستگاه

خروجی دوم، شاخص اصلی تقاضای بالقوه خواهد بود؛ زیرا اثر تفاوت تعداد پورت‌های ایستگاه‌ها را کنترل می‌کند و برای مقایسه و انتقال به نقاط کاندید در اصفهان مناسب‌تر است.

In [2]:
# Prompt: Build the leakage-free station-day charging-demand target from CHARGED hourly duration data.

def read_charged_csv(city_code, file_name):
    archive_path = f"{city_code}/{file_name}"

    with zipfile.ZipFile(CHARGED_ARCHIVE_PATH, "r") as charged_archive:
        with charged_archive.open(archive_path) as raw_file:
            return pd.read_csv(raw_file)


station_day_target_parts = []
target_audit_rows = []

with zipfile.ZipFile(CHARGED_ARCHIVE_PATH, "r") as charged_archive:
    for city in TRAINING_CITY_CODES:
        city_sites = read_charged_csv(city, "sites.csv").copy()

        site_identifier_column = (
            "site_id" if "site_id" in city_sites.columns else "site"
        )

        city_sites = (
            city_sites
            .rename(columns={site_identifier_column: "site_id"})
            [["site_id", "charger_num", "longitude", "latitude"]]
            .copy()
        )

        city_sites["site_id"] = city_sites["site_id"].astype(str)
        city_sites["charger_num"] = pd.to_numeric(
            city_sites["charger_num"],
            errors="coerce",
        )

        assert city_sites["charger_num"].notna().all()
        assert (city_sites["charger_num"] > 0).all()

        city_daily_parts = []
        archive_path = f"{city}/duration.csv"

        with charged_archive.open(archive_path) as raw_file:
            for duration_chunk in pd.read_csv(
                raw_file,
                chunksize=HOURLY_CHUNK_SIZE,
            ):
                timestamp_column = duration_chunk.columns[0]

                timestamps = pd.to_datetime(
                    duration_chunk[timestamp_column],
                    errors="coerce",
                )

                assert timestamps.notna().all()

                duration_values = (
                    duration_chunk
                    .iloc[:, 1:]
                    .apply(pd.to_numeric, errors="coerce")
                )

                assert duration_values.isna().sum().sum() == 0

                duration_values.index = timestamps.dt.normalize()

                daily_duration_wide = duration_values.groupby(
                    level=0
                ).sum()

                daily_duration_long = (
                    daily_duration_wide
                    .rename_axis("date")
                    .stack()
                    .rename("daily_duration_hours")
                    .reset_index()
                    .rename(columns={"level_1": "site_id"})
                )

                daily_duration_long["site_id"] = (
                    daily_duration_long["site_id"].astype(str)
                )

                city_daily_parts.append(daily_duration_long)

        city_station_day = (
            pd.concat(city_daily_parts, ignore_index=True)
            .groupby(["date", "site_id"], as_index=False)
            ["daily_duration_hours"]
            .sum()
        )

        city_station_day = city_station_day.merge(
            city_sites,
            on="site_id",
            how="left",
            validate="many_to_one",
        )

        assert city_station_day["charger_num"].notna().all()

        city_station_day["daily_duration_per_charger"] = (
            city_station_day["daily_duration_hours"]
            / city_station_day["charger_num"]
        )

        city_station_day["city_code"] = city
        city_station_day["city_name"] = CITY_LOOKUP[city]["city_name"]
        city_station_day["country_name"] = CITY_LOOKUP[city]["country_name"]

        station_day_target_parts.append(city_station_day)

        target_audit_rows.append(
            {
                "city_code": city,
                "station_day_records": len(city_station_day),
                "unique_dates": city_station_day["date"].nunique(),
                "unique_sites": city_station_day["site_id"].nunique(),
                "zero_daily_duration_percent": (
                    100
                    * (city_station_day["daily_duration_hours"] == 0).mean()
                ),
                "mean_daily_duration_hours": (
                    city_station_day["daily_duration_hours"].mean()
                ),
                "mean_daily_duration_per_charger": (
                    city_station_day["daily_duration_per_charger"].mean()
                ),
            }
        )

charged_station_day_targets = pd.concat(
    station_day_target_parts,
    ignore_index=True,
)

target_construction_audit = pd.DataFrame(target_audit_rows)

display(target_construction_audit)

TARGET_OUTPUT_PATH = (
    INTERIM_DATA_DIR / "charged_station_day_targets.csv.gz"
)

charged_station_day_targets.to_csv(
    TARGET_OUTPUT_PATH,
    index=False,
    compression="gzip",
    encoding="utf-8-sig",
)

target_construction_audit.to_csv(
    OUTPUT_TABLES_DIR / "charged_station_day_target_construction_audit.csv",
    index=False,
    encoding="utf-8-sig",
)

print(f"Station-day target shape: {charged_station_day_targets.shape}")
print(f"Saved target data: {TARGET_OUTPUT_PATH}")
print(
    "Saved audit: "
    "outputs/tables/charged_station_day_target_construction_audit.csv"
)

,city_code,station_day_records,unique_dates,unique_sites,zero_daily_duration_percent,mean_daily_duration_hours,mean_daily_duration_per_charger
0,AMS,448167,183,2449,55.1504,41.0308,26.2735
1,JHB,8601,183,47,26.9387,4.9596,3.9504
2,LOA,41907,183,229,3.0854,24.6045,11.2889
3,MEL,11529,183,63,3.7037,30.8167,30.2854
4,SPO,8601,183,47,14.3472,5.2868,4.9940


Station-day target shape: (518805, 10)
Saved target data: F:\UrbanEV_Charging_Demand\data\interim\charged_station_day_targets.csv.gz
Saved audit: outputs/tables/charged_station_day_target_construction_audit.csv


## ۲-۲. ساخت ویژگی‌های تقویمی و جوی در سطح ایستگاه-روز

ویژگی‌های تقویمی فقط از زمان‌نگار ثبت‌شده در دیتاست استخراج می‌شوند. چون منطقهٔ زمانی منابع مشخص نشده است، این ویژگی‌ها بیانگر «زمان ثبت‌شده» هستند و نباید به‌عنوان الگوی قطعیِ رفتار محلی تفسیر شوند.

داده‌های جوی ساعتی به سطح روز تجمیع می‌شوند. متغیر `preciptype` استفاده نمی‌شود و ویژگی `has_precipitation` از مجموع روزانهٔ `precip` ساخته خواهد شد.

In [3]:
# Prompt: Aggregate portable weather variables to daily level and merge them with station-day targets.

WEATHER_MEAN_VARIABLES = [
    "temp",
    "humidity",
    "windspeed",
    "visibility",
    "cloudcover",
]

WEATHER_SUM_VARIABLES = [
    "precip",
    "solarradiation",
]

daily_weather_parts = []

for city in TRAINING_CITY_CODES:
    city_weather = read_charged_csv(city, "weather.csv").copy()

    city_weather["timestamp"] = pd.to_datetime(
        city_weather["time"],
        errors="coerce",
    )

    assert city_weather["timestamp"].notna().all()

    required_weather_columns = (
        WEATHER_MEAN_VARIABLES + WEATHER_SUM_VARIABLES
    )

    for column_name in required_weather_columns:
        city_weather[column_name] = pd.to_numeric(
            city_weather[column_name],
            errors="coerce",
        )

    assert city_weather[required_weather_columns].isna().sum().sum() == 0

    city_weather["date"] = city_weather["timestamp"].dt.normalize()

    city_daily_weather = (
        city_weather
        .groupby("date", as_index=False)
        .agg(
            temp_mean=("temp", "mean"),
            humidity_mean=("humidity", "mean"),
            windspeed_mean=("windspeed", "mean"),
            visibility_mean=("visibility", "mean"),
            cloudcover_mean=("cloudcover", "mean"),
            precip_total=("precip", "sum"),
            solarradiation_total=("solarradiation", "sum"),
        )
    )

    city_daily_weather["has_precipitation"] = (
        city_daily_weather["precip_total"] > 0
    ).astype(int)

    city_daily_weather["city_code"] = city
    daily_weather_parts.append(city_daily_weather)

charged_daily_weather = pd.concat(
    daily_weather_parts,
    ignore_index=True,
)

charged_station_day_temporal = charged_station_day_targets.merge(
    charged_daily_weather,
    on=["city_code", "date"],
    how="left",
    validate="many_to_one",
)

assert (
    charged_station_day_temporal[
        [
            "temp_mean",
            "humidity_mean",
            "windspeed_mean",
            "visibility_mean",
            "cloudcover_mean",
            "precip_total",
            "solarradiation_total",
            "has_precipitation",
        ]
    ]
    .isna()
    .sum()
    .sum()
    == 0
)

charged_station_day_temporal["recorded_month"] = (
    charged_station_day_temporal["date"].dt.month
)

charged_station_day_temporal["recorded_day_of_week"] = (
    charged_station_day_temporal["date"].dt.dayofweek
)

charged_station_day_temporal["recorded_day_of_year"] = (
    charged_station_day_temporal["date"].dt.dayofyear
)

charged_station_day_temporal["month_sin"] = np.sin(
    2 * np.pi * charged_station_day_temporal["recorded_month"] / 12
)

charged_station_day_temporal["month_cos"] = np.cos(
    2 * np.pi * charged_station_day_temporal["recorded_month"] / 12
)

TEMPORAL_OUTPUT_PATH = (
    INTERIM_DATA_DIR / "charged_station_day_temporal_features.csv.gz"
)

charged_station_day_temporal.to_csv(
    TEMPORAL_OUTPUT_PATH,
    index=False,
    compression="gzip",
    encoding="utf-8-sig",
)

weather_merge_audit = (
    charged_station_day_temporal
    .groupby("city_code", as_index=False)
    .agg(
        station_day_records=("site_id", "size"),
        unique_dates=("date", "nunique"),
        precipitation_days=("has_precipitation", "sum"),
        mean_temperature=("temp_mean", "mean"),
    )
)

display(weather_merge_audit)

print(f"Temporal-feature shape: {charged_station_day_temporal.shape}")
print(f"Saved temporal features: {TEMPORAL_OUTPUT_PATH}")

,city_code,station_day_records,unique_dates,precipitation_days,mean_temperature
0,AMS,448167,183,262043,15.8283
1,JHB,8601,183,1128,14.6361
2,LOA,41907,183,4122,19.9141
3,MEL,11529,183,8127,12.8880
4,SPO,8601,183,1598,18.9889


Temporal-feature shape: (518805, 23)
Saved temporal features: F:\UrbanEV_Charging_Demand\data\interim\charged_station_day_temporal_features.csv.gz


In [4]:
weather_city_summary = (
    charged_daily_weather
    .groupby("city_code", as_index=False)
    .agg(
        unique_dates=("date", "nunique"),
        precipitation_days=("has_precipitation", "sum"),
        mean_temperature=("temp_mean", "mean"),
    )
)

station_day_record_counts = (
    charged_station_day_temporal
    .groupby("city_code", as_index=False)
    .size()
    .rename(columns={"size": "station_day_records"})
)

weather_merge_audit = (
    station_day_record_counts
    .merge(weather_city_summary, on="city_code", how="left")
    .sort_values("city_code")
)

display(weather_merge_audit)

print(f"Temporal-feature shape: {charged_station_day_temporal.shape}")
print(f"Saved temporal features: {TEMPORAL_OUTPUT_PATH}")

,city_code,station_day_records,unique_dates,precipitation_days,mean_temperature
0,AMS,448167,183,107,15.8283
1,JHB,8601,183,24,14.6361
2,LOA,41907,183,18,19.9141
3,MEL,11529,183,129,12.8880
4,SPO,8601,183,34,18.9889


Temporal-feature shape: (518805, 23)
Saved temporal features: F:\UrbanEV_Charging_Demand\data\interim\charged_station_day_temporal_features.csv.gz


## ۲-۳. استانداردسازی مفهومی نقاط مورد علاقه

انواع خام POI به گروه‌های مفهومیِ قابل‌انتقال تبدیل می‌شوند. فقط گروه‌هایی استفاده خواهند شد که بتوان آن‌ها را با همان منطق از داده‌های UrbanEV و OpenStreetMap شهر اصفهان استخراج کرد.

دسته‌های خام `other` و نوع‌های تعریف‌نشده مستقیماً وارد مدل نخواهند شد. تمام نگاشت‌ها به‌صورت صریح ثبت می‌شوند تا قابل ممیزی و تکرار باشند.

In [5]:
# Prompt: Map raw POI types to auditable and transferable semantic groups.

POI_GROUP_DEFINITIONS = {
    "parking_mobility": {
        "parking",
        "parking_space",
        "parking_entrance",
        "bicycle_parking",
        "bus_station",
        "bus_stop",
        "taxi",
        "fuel",
        "car_rental",
        "car_wash",
        "charging_station",
    },
    "education": {
        "school",
        "university",
        "college",
        "kindergarten",
        "library",
    },
    "food_commerce": {
        "restaurant",
        "fast_food",
        "cafe",
        "bar",
        "pub",
        "marketplace",
        "supermarket",
        "convenience",
        "bank",
        "atm",
        "mall",
    },
    "healthcare": {
        "hospital",
        "clinic",
        "doctors",
        "dentist",
        "pharmacy",
        "veterinary",
    },
    "public_services": {
        "fire_station",
        "police",
        "post_office",
        "post_box",
        "townhall",
        "courthouse",
        "place_of_worship",
        "toilets",
        "drinking_water",
        "waste_basket",
        "recycling",
    },
    "recreation": {
        "park",
        "playground",
        "sports_centre",
        "stadium",
        "swimming_pool",
        "cinema",
        "theatre",
        "museum",
        "bench",
        "shelter",
    },
}

poi_type_to_group = {
    poi_type: group_name
    for group_name, poi_types in POI_GROUP_DEFINITIONS.items()
    for poi_type in poi_types
}

assert len(poi_type_to_group) == sum(
    len(poi_types)
    for poi_types in POI_GROUP_DEFINITIONS.values()
)

poi_group_tables = []
poi_unmapped_tables = []

for city in TRAINING_CITY_CODES:
    city_poi = read_charged_csv(city, "poi.csv").copy()

    city_poi["poi_type"] = (
        city_poi["type"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    city_poi["poi_group"] = city_poi["poi_type"].map(
        poi_type_to_group
    )

    city_poi["poi_group"] = city_poi["poi_group"].fillna(
        "unmapped_or_other"
    )

    city_group_counts = (
        city_poi
        .groupby("poi_group", as_index=False)
        .size()
        .rename(columns={"size": "poi_count"})
    )

    city_group_counts.insert(0, "city_code", city)
    poi_group_tables.append(city_group_counts)

    city_unmapped_counts = (
        city_poi
        .loc[city_poi["poi_group"] == "unmapped_or_other", "poi_type"]
        .value_counts()
        .rename_axis("poi_type")
        .reset_index(name="poi_count")
    )

    city_unmapped_counts.insert(0, "city_code", city)
    poi_unmapped_tables.append(city_unmapped_counts)

poi_group_distribution = pd.concat(
    poi_group_tables,
    ignore_index=True,
)

poi_unmapped_distribution = pd.concat(
    poi_unmapped_tables,
    ignore_index=True,
)

print("POI counts by semantic group and city:")
display(
    poi_group_distribution
    .pivot(
        index="poi_group",
        columns="city_code",
        values="poi_count",
    )
    .fillna(0)
    .astype(int)
)

print("Most frequent unmapped POI types:")
display(
    poi_unmapped_distribution
    .groupby("poi_type", as_index=False)["poi_count"]
    .sum()
    .sort_values("poi_count", ascending=False)
    .head(25)
)

poi_group_distribution.to_csv(
    OUTPUT_TABLES_DIR / "charged_poi_semantic_group_distribution.csv",
    index=False,
    encoding="utf-8-sig",
)

poi_unmapped_distribution.to_csv(
    OUTPUT_TABLES_DIR / "charged_unmapped_poi_distribution.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Saved POI standardization audit tables in outputs/tables/")

POI counts by semantic group and city:


city_code,AMS,JHB,LOA,MEL,SPO
poi_group,,,,,
education,458,794,1316,2532,1791
food_commerce,3734,1700,4832,10478,5496
healthcare,278,359,522,2221,1642
parking_mobility,22775,2161,8395,57676,10712
public_services,4073,620,2265,8960,1929
recreation,4309,235,1197,9129,566
unmapped_or_other,15713,41619,24255,526810,27583


Most frequent unmapped POI types:


,poi_type,poi_count
169,other,626380
259,telephone,1008
18,bbq,749
236,social_facility,637
273,vending_machine,633
21,bicycle_rental,606
59,community_centre,601
48,childcare,503
104,fountain,448
275,waste_disposal,445


Saved POI standardization audit tables in outputs/tables/


## ۲-۴. ساخت ویژگی‌های دسترسی مکانی به خدمات شهری

برای هر ایستگاه، تعداد POIهای هر گروه مفهومی در دو شعاع ۵۰۰ و ۱۰۰۰ متری محاسبه می‌شود. این ویژگی‌ها نمایندهٔ تراکم خدمات و جاذب‌های سفر در پیرامون ایستگاه هستند.

فاصله‌ها با روش haversine محاسبه می‌شوند تا اثر انحنای سطح زمین در مختصات جغرافیایی لحاظ شود. همین روش و همین شعاع‌ها بعداً بدون تغییر برای UrbanEV و اصفهان اجرا خواهند شد.

In [6]:
# Prompt: Generate transferable POI accessibility features around each charging station.

from sklearn.neighbors import BallTree

EARTH_RADIUS_METERS = 6_371_000
SPATIAL_RADII_METERS = [500, 1000]
SPATIAL_POI_GROUPS = list(POI_GROUP_DEFINITIONS.keys())

station_locations = (
    charged_station_day_temporal[
        [
            "city_code",
            "site_id",
            "longitude",
            "latitude",
            "charger_num",
        ]
    ]
    .drop_duplicates()
    .copy()
)

spatial_feature_tables = []

for city in TRAINING_CITY_CODES:
    city_stations = (
        station_locations
        .loc[station_locations["city_code"] == city]
        .reset_index(drop=True)
        .copy()
    )

    city_poi = read_charged_csv(city, "poi.csv").copy()

    city_poi["poi_type"] = (
        city_poi["type"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    city_poi["poi_group"] = (
        city_poi["poi_type"]
        .map(poi_type_to_group)
        .fillna("unmapped_or_other")
    )

    station_coordinates_radians = np.radians(
        city_stations[["latitude", "longitude"]].to_numpy()
    )

    city_spatial_features = city_stations[
        ["city_code", "site_id"]
    ].copy()

    for poi_group in SPATIAL_POI_GROUPS:
        group_poi = city_poi.loc[
            city_poi["poi_group"] == poi_group,
            ["latitude", "longitude"],
        ].dropna()

        poi_coordinates_radians = np.radians(
            group_poi.to_numpy()
        )

        poi_tree = BallTree(
            poi_coordinates_radians,
            metric="haversine",
        )

        for radius_meters in SPATIAL_RADII_METERS:
            feature_name = (
                f"poi_{poi_group}_within_{radius_meters}m"
            )

            city_spatial_features[feature_name] = (
                poi_tree.query_radius(
                    station_coordinates_radians,
                    r=radius_meters / EARTH_RADIUS_METERS,
                    count_only=True,
                )
            )

    spatial_feature_tables.append(city_spatial_features)

charged_site_spatial_features = pd.concat(
    spatial_feature_tables,
    ignore_index=True,
)

spatial_feature_columns = [
    column_name
    for column_name in charged_site_spatial_features.columns
    if column_name.startswith("poi_")
]

assert (
    charged_site_spatial_features[spatial_feature_columns]
    .isna()
    .sum()
    .sum()
    == 0
)

spatial_feature_audit = (
    charged_site_spatial_features
    .groupby("city_code", as_index=False)
    .agg(
        station_count=("site_id", "nunique"),
        mean_poi_within_500m=(
            "poi_parking_mobility_within_500m",
            "mean",
        ),
        mean_poi_within_1000m=(
            "poi_parking_mobility_within_1000m",
            "mean",
        ),
    )
)

display(spatial_feature_audit)

SPATIAL_OUTPUT_PATH = (
    INTERIM_DATA_DIR / "charged_site_spatial_features.csv.gz"
)

charged_site_spatial_features.to_csv(
    SPATIAL_OUTPUT_PATH,
    index=False,
    compression="gzip",
    encoding="utf-8-sig",
)

print(f"Spatial-feature shape: {charged_site_spatial_features.shape}")
print(f"Saved spatial features: {SPATIAL_OUTPUT_PATH}")

,city_code,station_count,mean_poi_within_500m,mean_poi_within_1000m
0,AMS,2449,80.1878,305.0988
1,JHB,47,9.1064,19.4043
2,LOA,229,11.3668,33.4803
3,MEL,63,54.8095,130.6190
4,SPO,47,33.6809,139.5319


Spatial-feature shape: (2835, 14)
Saved spatial features: F:\UrbanEV_Charging_Demand\data\interim\charged_site_spatial_features.csv.gz


## ۲-۵. ساخت مجموعه‌دادهٔ پایهٔ مدل‌سازی

ویژگی‌های زمانی، جوی و مکانی در یک مجموعه‌دادهٔ پایه ادغام می‌شوند. این مجموعه‌داده هنوز شامل همهٔ ویژگی‌های بالقوه است؛ انتخاب نهایی متغیرها فقط پس از ممیزی و همسان‌سازی دیتاست مستقل UrbanEV انجام خواهد شد.

مختصات، شناسهٔ ایستگاه و نام شهر صرفاً برای اتصال داده، ممیزی و تحلیل فضایی نگهداری می‌شوند و به‌عنوان ورودی مستقیم مدل استفاده نخواهند شد.

In [7]:
# Prompt: Merge temporal and spatial features into a leakage-free CHARGED modeling-base dataset.

charged_modeling_base = charged_station_day_temporal.merge(
    charged_site_spatial_features,
    on=["city_code", "site_id"],
    how="left",
    validate="many_to_one",
)

spatial_feature_columns = [
    column_name
    for column_name in charged_site_spatial_features.columns
    if column_name.startswith("poi_")
]

assert (
    charged_modeling_base[spatial_feature_columns]
    .isna()
    .sum()
    .sum()
    == 0
)

charged_modeling_base["target_log_daily_duration_per_charger"] = np.log1p(
    charged_modeling_base["daily_duration_per_charger"]
)

metadata_columns = [
    "city_code",
    "city_name",
    "country_name",
    "site_id",
    "longitude",
    "latitude",
    "date",
]

target_columns = [
    "daily_duration_hours",
    "daily_duration_per_charger",
    "target_log_daily_duration_per_charger",
]

candidate_predictor_columns = [
    "charger_num",
    "temp_mean",
    "humidity_mean",
    "windspeed_mean",
    "visibility_mean",
    "cloudcover_mean",
    "precip_total",
    "solarradiation_total",
    "has_precipitation",
    "recorded_month",
    "recorded_day_of_week",
    "recorded_day_of_year",
    "month_sin",
    "month_cos",
] + spatial_feature_columns

modeling_base_columns = (
    metadata_columns
    + target_columns
    + candidate_predictor_columns
)

charged_modeling_base = charged_modeling_base[
    modeling_base_columns
].copy()

assert charged_modeling_base.duplicated(
    subset=["city_code", "site_id", "date"]
).sum() == 0

assert (
    charged_modeling_base[candidate_predictor_columns]
    .isna()
    .sum()
    .sum()
    == 0
)

modeling_base_audit = (
    charged_modeling_base
    .groupby("city_code", as_index=False)
    .agg(
        station_day_records=("site_id", "size"),
        unique_sites=("site_id", "nunique"),
        unique_dates=("date", "nunique"),
        mean_target=(
            "daily_duration_per_charger",
            "mean",
        ),
        median_target=(
            "daily_duration_per_charger",
            "median",
        ),
        zero_target_percent=(
            "daily_duration_per_charger",
            lambda series: 100 * (series == 0).mean(),
        ),
    )
)

display(modeling_base_audit)

MODELING_BASE_PATH = (
    PROCESSED_DATA_DIR / "charged_modeling_base.csv.gz"
)

charged_modeling_base.to_csv(
    MODELING_BASE_PATH,
    index=False,
    compression="gzip",
    encoding="utf-8-sig",
)

modeling_base_audit.to_csv(
    OUTPUT_TABLES_DIR / "charged_modeling_base_audit.csv",
    index=False,
    encoding="utf-8-sig",
)

print(f"Modeling-base shape: {charged_modeling_base.shape}")
print(f"Candidate predictor count: {len(candidate_predictor_columns)}")
print(f"Saved modeling base: {MODELING_BASE_PATH}")

,city_code,station_day_records,unique_sites,unique_dates,mean_target,median_target,zero_target_percent
0,AMS,448167,2449,183,26.2735,0.0000,55.1504
1,JHB,8601,47,183,3.9504,2.5349,26.9387
2,LOA,41907,229,183,11.2889,8.7500,3.0854
3,MEL,11529,63,183,30.2854,22.0000,3.7037
4,SPO,8601,47,183,4.9940,2.6316,14.3472


Modeling-base shape: (518805, 36)
Candidate predictor count: 26
Saved modeling base: F:\UrbanEV_Charging_Demand\data\processed\charged_modeling_base.csv.gz
